# Does learning new tasks damage older abilities?

**This is one training experiment. There are no A/B runs and no forgetting requests.**

We start from the saved model that already learned tasks 3 and 0. Then we train
it on tasks 9, 5, and 17, in that order. After each task, we test all tasks learned
so far. The result is one accuracy table.

Accuracy is the percentage of test images answered correctly. If task 0 starts
at 44% and later falls to 30%, it has lost 14 percentage points. Those numbers
are an example, not today's result.

Use a fresh Colab session with an A100 GPU. Run each code cell from top to bottom
and wait for it to finish. The last cell saves Drive and ends the GPU session.

This notebook uses an already published version of the training code. It does
not need the proposed `exact_math()` or `SpatialMean` changes. It does not
measure an exact-match replay. Keep the previous replay results on Drive.

A checkpoint is a saved model file. Our starting file is **before** the operation
that forgot task 3. If the session ends early, completed tasks remain saved.
You can reconnect and rerun this notebook with the same experiment name to
continue from the last completed task. An interrupted task will start again.


## 1. Connect Drive and load the training code

Choose A100 in Colab's runtime settings. Run this cell and allow access to your
Google Drive when asked. The cell downloads a fixed version of the project code,
then prints the code version. You do not need to change that version.

If this cell shows an error, stop and copy the message. Do not run the later cells.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
REPO = Path("/content/uncle-learning-only")
# This published version contains the original ResNet pooling and training loop.
CODE_VERSION = "83e1c32773fc59886c8b43a0ed159ccf6424637a"
assert not any(name == "uncle" or name.startswith("uncle.") for name in sys.modules), "Start a new Colab session before loading the code again."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "Use a new Colab session. This folder does not contain the expected project."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "Use a new Colab session. This is a different project."
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Save your code changes and use a new Colab session."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Code version:", CODE_VERSION)


## 2. Check the starting file and choose where to save

The starting file below is `before_forgetting.pt`. It must contain the model
after learning task 3 and task 0, with no forgetting operation applied.

`EXPERIMENT_ID` is the name of the folder for this experiment. Keep it unchanged
if you reconnect to continue this run. For a separate new experiment, choose a
new name. The original starting model will not be overwritten.

Run the cell. It prints the saved scores, the learning settings, and the GPU name.
These saved scores will be measured again in step 4 before new training begins.


In [ ]:
from dataclasses import replace
import json
import torch
from uncle import checkpoint as checkpointing
from uncle.config import Config

EXPERIMENT_ID = "20261005_learning_only_01"
ROOT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace")
SOURCE = ROOT / "before_forgetting.pt"
OUTPUT = ROOT / "learning_only" / EXPERIMENT_ID
DATA = Path("/content/data/tiny-imagenet-200")
TASK_ORDER = ("3", "0", "9", "5", "17")

assert torch.cuda.is_available(), "Choose a GPU in Colab's runtime settings."
gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
assert gpu_name == "NVIDIA A100-SXM4-40GB", f"This run uses A100. Colab provided {gpu_name}; choose A100 in a new session."
assert SOURCE.is_file(), f"Starting file not found: {SOURCE}. Check its location on Drive."
source = checkpointing.load(SOURCE)
assert [(r["action"], r["task"]) for r in source["history"]] == [("learn", "3"), ("learn", "0")], "Wrong starting file: it must end after learning tasks 3 and 0."
assert not source["forgotten"], "The starting file must have no forgotten tasks."
config = replace(Config(**source["config"]), requests=tuple(("learn", task) for task in TASK_ORDER))
assert config.dataset == "tiny_imagenet" and config.backbone == "resnet50"
assert config.torch_device.type == "cuda", "The saved model must use a GPU configuration."
print("GPU:", gpu_name)
print("Saved task scores:", source["previous"])
print("Learning rate:", config.learning_rate, "Protection strength (beta):", config.beta)
print("Passes through each task's images:", config.epochs)
print("Results folder:", OUTPUT)
del source


## 3. Prepare the images and the recording code

Run both cells below. The first cell downloads the images if needed. The second
cell defines the instructions for training and saving. It does not start training.

We keep the starting model's learning settings unchanged. We also keep the part
of training intended to protect older tasks. There is no new settings search.


In [ ]:
from uncle.streams import build_tasks

tasks = build_tasks(config, root=DATA, include=TASK_ORDER, download=True)
print("Images ready for tasks:", ", ".join(TASK_ORDER))


In [ ]:
import hashlib
import random
import time
import numpy as np
from uncle.experiment import validate_starting_checkpoint
from uncle.hypernet import HyperNetwork, build_target
from uncle.telemetry import environment
from uncle.trainer import UnCLe


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".writing")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def show_table(rows, task_order):
    print("Accuracy (% of test images answered correctly)")
    print(f"{'Stage':25s}" + "".join(f"{'Task ' + task:>10s}" for task in task_order))
    for row in rows:
        values = [f"{row['accuracies'][task]:.1f}" if task in row['accuracies'] else "" for task in task_order]
        print(f"{row['stage']:25s}" + "".join(f"{value:>10s}" for value in values))


def run_learning_only(config, tasks, source_path, output, *, progress=True):
    output, source_path = Path(output), Path(source_path)
    task_order = tuple(task for action, task in config.requests)
    if any(action != "learn" for action, task in config.requests):
        raise ValueError("This experiment must contain only learning requests.")
    source = checkpointing.load(source_path)
    if source is None:
        raise FileNotFoundError(source_path)
    validate_starting_checkpoint(source, config)
    if source["forgotten"]:
        raise ValueError("The starting model must have no forgotten tasks.")
    source_done = len(source["history"])
    output.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output / "checkpoint.pt"
    identity = {"source_sha256": file_hash(source_path),
                "config": json.loads(json.dumps(vars(config))),
                "code_version": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()}
    origin_path = output / "origin.json"
    if origin_path.exists():
        if json.loads(origin_path.read_text()) != identity:
            raise ValueError("This results folder belongs to different code, settings, or a starting file. Use a new experiment name.")
    else:
        if any(output.iterdir()):
            raise ValueError("This results folder already has files without a starting record. Use a new experiment name.")
        save_json(origin_path, identity)
    saved = checkpointing.load(checkpoint_path, config) if checkpoint_path.exists() else source
    del source
    if saved["forgotten"]:
        raise ValueError("The saved run contains a forgetting operation.")
    expected = list(config.requests[:len(saved["history"])])
    if [(r["action"], r["task"]) for r in saved["history"]] != expected:
        raise ValueError("Saved task order differs from this experiment.")
    torch.manual_seed(config.seed)
    random.seed(config.seed)
    np.random.seed(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=progress)
    done = checkpointing.restore(saved, hypernet=hypernet, uncle=trainer)
    history = list(saved["history"])
    seen = list(saved["seen"])
    costs = list(saved["costs"])
    setup_seconds = saved["setup_seconds"]
    del saved
    # Keep a separate record for each connection; do not overwrite the first one.
    session = environment(config, hypernet, tasks)
    session["completed_requests_at_connection"] = done
    session["calculation_settings"] = {
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
        "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    }
    save_json(output / f"session_{time.time_ns()}.json", session)
    baseline_path = output / "starting_scores.json"
    if baseline_path.exists():
        baseline = json.loads(baseline_path.read_text())
    else:
        if done != source_done:
            raise ValueError("Starting scores are missing. Keep these files and inspect them before continuing.")
        devices = list(range(torch.cuda.device_count())) if config.torch_device.type == "cuda" else []
        with torch.random.fork_rng(devices=devices):
            baseline = {task: trainer.accuracy(task) for task in seen}
        save_json(baseline_path, baseline)
    rows = [{"stage": "Starting model", "accuracies": baseline}]
    rows.extend({"stage": f"After learning {r['task']}", "accuracies": r["after"]} for r in history[source_done:])

    def save_report():
        report = {"complete": len(history) == len(config.requests),
                  "task_order": list(task_order), "rows": rows}
        save_json(output / "accuracy_table.json", report)
        lines = ["stage," + ",".join("task_" + task for task in task_order)]
        for row in rows:
            lines.append(row["stage"] + "," + ",".join(str(row["accuracies"].get(task, "")) for task in task_order))
        (output / "accuracy_table.csv").write_text("\n".join(lines) + "\n", encoding="utf-8")
        show_table(rows, task_order)
        return report

    report = save_report()
    for index, (action, task) in enumerate(config.requests):
        if index < done:
            continue
        before = dict(rows[-1]["accuracies"])
        began = time.perf_counter()
        print("Learning task", task, flush=True)
        losses = trainer.learn(task, [name for name in seen if name != task])
        seen.append(task)
        devices = list(range(torch.cuda.device_count())) if config.torch_device.type == "cuda" else []
        with torch.random.fork_rng(devices=devices):
            after = {name: trainer.accuracy(name) for name in seen}
        history.append({"index": index, "action": "learn", "task": task,
                        "before": before, "after": after, "seen": list(seen),
                        "forgotten": [], "final_loss": losses[-1], "burn_in": None})
        costs.append({"action": "learn", "task": task, "seconds": time.perf_counter() - began})
        checkpointing.save(checkpoint_path, config=config, hypernet=hypernet,
                           uncle=trainer, history=history, seen=seen, forgotten=[],
                           previous=after, costs=costs, setup_seconds=setup_seconds)
        rows.append({"stage": f"After learning {task}", "accuracies": after})
        report = save_report()
        print("Task", task, "saved. A disconnected session can continue from here.", flush=True)
    if file_hash(source_path) != identity["source_sha256"]:
        raise ValueError("The original starting file changed during this run.")
    return report


## 4. Run the experiment

Run this cell. It first prints the starting accuracy table, then trains tasks
9, 5, and 17. A new row appears after each task. Blank entries mean that task
has not been learned yet.

The notebook saves after each completed task. If Colab disconnects, reconnect
and run the notebook again with the same `EXPERIMENT_ID`. Completed tasks are
skipped. No new experiment name is needed for continuing this run.

When the final row appears, run step 5 immediately to finish saving Drive.


In [ ]:
report = run_learning_only(config, tasks, SOURCE, OUTPUT)
assert report["complete"], "Training has not completed all three new tasks."
print("Training finished. Results saved in:", OUTPUT)


## 5. Verify the results and finish saving Drive

Run this cell. It checks the final saved model and table, prints the table once
more, finishes saving Drive, and releases the GPU. Copy the printed table before
closing the notebook, or keep `accuracy_table.csv` from the results folder.

Share that table with me. We will look for the first new task that causes an
older task's accuracy to fall. This single run helps choose our next test; it
cannot establish that the result holds across other seeds or task orders.


In [ ]:
from google.colab import runtime

saved_report = json.loads((OUTPUT / "accuracy_table.json").read_text())
assert saved_report == report and saved_report["complete"]
final = checkpointing.load(OUTPUT / "checkpoint.pt", config)
assert [(r["action"], r["task"]) for r in final["history"]] == list(config.requests)
assert not final["forgotten"]
assert final["previous"] == saved_report["rows"][-1]["accuracies"]
del final
show_table(saved_report["rows"], TASK_ORDER)
print("Files checked. Saving Drive and ending the GPU session.", flush=True)
drive.flush_and_unmount(timeout_ms=600000)
runtime.unassign()
